In [1]:
from pycoingecko import CoinGeckoAPI
import pandas as pd

cg = CoinGeckoAPI()

def get_ohlc(coin_id, vs="usd", days="max"):
    """
    Fetches historical crypto data and returns a clean Daily OHLC DataFrame:
    - id
    - name
    - timestamp
    - open
    - high
    - low
    - close
    - volume
    """

    # ---- Fetch coin info to get name ----
    coin_info = cg.get_coin_by_id(coin_id)
    coin_name = coin_info["name"]

    # ---- Fetch raw market data ----
    data = cg.get_coin_market_chart_by_id(
        id=coin_id,
        vs_currency=vs,
        days=days
    )

    # Convert lists into DataFrames
    prices = pd.DataFrame(data["prices"], columns=["timestamp", "price"])
    prices["timestamp"] = pd.to_datetime(prices["timestamp"], unit="ms")

    market_caps = pd.DataFrame(data["market_caps"], columns=["timestamp", "market_cap"])
    market_caps["timestamp"] = pd.to_datetime(market_caps["timestamp"], unit="ms")

    volumes = pd.DataFrame(data["total_volumes"], columns=["timestamp", "volume"])
    volumes["timestamp"] = pd.to_datetime(volumes["timestamp"], unit="ms")

    # Merge everything
    df = prices.merge(market_caps, on="timestamp").merge(volumes, on="timestamp")
    df.set_index("timestamp", inplace=True)

    # ---- Build daily OHLC ----
    ohlc = df["price"].resample("1D").agg({
        "open": "first",
        "high": "max",
        "low":  "min",
        "close": "last"
    })

    # Daily volume
    ohlc["volume"] = df["volume"].resample("1D").sum()

    # Clean NaNs
    ohlc.dropna(inplace=True)

    # Reset index for final output
    ohlc = ohlc.reset_index()

    # ---- Add ID & Name columns ----
    ohlc["id"] = coin_id
    ohlc["name"] = coin_name

    # Reorder columns
    ohlc = ohlc[["id", "name", "timestamp", "open", "high", "low", "close", "volume"]]

    return ohlc


In [2]:
coins = [
    {"name":"Bitcoin","id":"bitcoin","e":0,"s":1,"g":0,"esg":1},
    {"name":"Ethereum","id":"ethereum","e":1,"s":1,"g":1,"esg":3},
    {"name":"Cardano","id":"cardano","e":1,"s":1,"g":1,"esg":3},
    {"name":"Solana","id":"solana","e":1,"s":1,"g":1,"esg":3},
    {"name":"Polkadot","id":"polkadot","e":1,"s":1,"g":1,"esg":3},
    {"name":"Algorand","id":"algorand","e":1,"s":1,"g":1,"esg":3},
    {"name":"Tezos","id":"tezos","e":1,"s":1,"g":1,"esg":3},
    {"name":"Avalanche","id":"avalanche-2","e":1,"s":1,"g":1,"esg":3},
    {"name":"NEAR","id":"near","e":1,"s":1,"g":1,"esg":3},
    {"name":"Hedera","id":"hedera-hashgraph","e":1,"s":1,"g":0,"esg":2},
    {"name":"Stellar","id":"stellar","e":1,"s":1,"g":0,"esg":2},
    {"name":"Celo","id":"celo","e":1,"s":1,"g":1,"esg":3},
    {"name":"Ripple","id":"ripple","e":1,"s":1,"g":0,"esg":2},
    {"name":"Nano","id":"nano","e":1,"s":1,"g":0,"esg":2},
    {"name":"IOTA","id":"iota","e":1,"s":1,"g":0,"esg":2},
    {"name":"Energy Web Token","id":"energy-web-token","e":1,"s":1,"g":1,"esg":3},
    {"name":"Power Ledger","id":"power-ledger","e":1,"s":1,"g":0,"esg":2},
    {"name":"Regen Network","id":"regen","e":1,"s":1,"g":1,"esg":3},
    {"name":"KlimaDAO","id":"klima-dao","e":1,"s":1,"g":1,"esg":3},
    {"name":"Moss Carbon Credit","id":"moss-carbon-credit","e":1,"s":1,"g":1,"esg":3},
    {"name":"IMPT","id":"impt","e":1,"s":1,"g":1,"esg":3},
    {"name":"SolarCoin","id":"solarcoin","e":1,"s":1,"g":0,"esg":2},
    {"name":"Peercoin","id":"peercoin","e":1,"s":0,"g":0,"esg":1},
    {"name":"Flow","id":"flow","e":1,"s":1,"g":0,"esg":2},
    {"name":"WAX","id":"wax","e":1,"s":1,"g":1,"esg":3},
    {"name":"VeChain","id":"vechain","e":1,"s":1,"g":0,"esg":2},
    {"name":"Chia","id":"chia","e":1,"s":0,"g":0,"esg":1},
    {"name":"Signum","id":"signum","e":1,"s":0,"g":0,"esg":1},
    {"name":"Telcoin","id":"telcoin","e":1,"s":1,"g":0,"esg":2},
    {"name":"Electroneum","id":"electroneum","e":1,"s":1,"g":0,"esg":2},
    {"name":"Worldcoin","id":"worldcoin-wld","e":1,"s":1,"g":0,"esg":2},
    {"name":"GoodDollar","id":"gooddollar","e":1,"s":1,"g":1,"esg":3},
    {"name":"VitaDAO","id":"vitadao","e":1,"s":1,"g":1,"esg":3},
    {"name":"Civic","id":"civic","e":1,"s":1,"g":0,"esg":2},
    {"name":"Decred","id":"decred","e":0,"s":1,"g":1,"esg":2},
    {"name":"Ethereum Name Service","id":"ethereum-name-service","e":1,"s":1,"g":1,"esg":3},
    {"name":"Cosmos","id":"cosmos","e":1,"s":1,"g":1,"esg":3},
    {"name":"Kava","id":"kava","e":1,"s":1,"g":1,"esg":3},
    {"name":"Gnosis","id":"gnosis","e":1,"s":1,"g":1,"esg":3},
    {"name":"Maker","id":"maker","e":1,"s":1,"g":1,"esg":3},
    {"name":"Polygon","id":"polygon","e":1,"s":1,"g":1,"esg":3},
    {"name":"5ireChain","id":"5ire","e":1,"s":1,"g":1,"esg":3},
    {"name":"SocialGood","id":"socialgood","e":1,"s":1,"g":0,"esg":2},
    {"name":"TRON","id":"tron","e":1,"s":0,"g":1,"esg":2},
    {"name":"Ethereum Classic","id":"ethereum-classic","e":0,"s":0,"g":0,"esg":0},
    {"name":"Bitcoin Cash","id":"bitcoin-cash","e":0,"s":1,"g":0,"esg":1},
    {"name":"Litecoin","id":"litecoin","e":0,"s":1,"g":0,"esg":1},
    {"name":"Dogecoin","id":"dogecoin","e":0,"s":1,"g":0,"esg":1},
    {"name":"Monero","id":"monero","e":0,"s":1,"g":0,"esg":1},
    {"name":"Zcash","id":"zcash","e":0,"s":1,"g":0,"esg":1},
    {"name":"Dash","id":"dash","e":0,"s":1,"g":1,"esg":2},
    {"name":"Shiba Inu","id":"shiba-inu","e":1,"s":1,"g":1,"esg":3},
    {"name":"Pepe","id":"pepe","e":1,"s":0,"g":0,"esg":1},
    {"name":"Floki","id":"floki","e":1,"s":1,"g":1,"esg":3},
    {"name":"HEX","id":"hex","e":1,"s":0,"g":0,"esg":1},
    {"name":"Terra Classic","id":"terra-luna-classic","e":1,"s":0,"g":1,"esg":2},
    {"name":"ApeCoin","id":"apecoin","e":1,"s":1,"g":1,"esg":3},
    {"name":"Axie Infinity","id":"axie-infinity","e":1,"s":1,"g":1,"esg":3},
    {"name":"Decentraland","id":"decentraland","e":1,"s":1,"g":1,"esg":3},
    {"name":"The Sandbox","id":"the-sandbox","e":1,"s":1,"g":1,"esg":3},
    {"name":"Enjin Coin","id":"enjincoin","e":1,"s":1,"g":0,"esg":2},
    {"name":"Gala","id":"gala","e":1,"s":1,"g":0,"esg":2},
    {"name":"Binance Coin","id":"binancecoin","e":1,"s":0,"g":0,"esg":1},
    {"name":"OKB","id":"okb","e":1,"s":0,"g":0,"esg":1},
    {"name":"Huobi Token","id":"huobi-token","e":1,"s":0,"g":0,"esg":1},
    {"name":"KuCoin Token","id":"kucoin-shares","e":1,"s":0,"g":0,"esg":1},
    {"name":"Tether","id":"tether","e":1,"s":1,"g":0,"esg":2},
    {"name":"USD Coin","id":"usd-coin","e":1,"s":1,"g":0,"esg":2},
    {"name":"Binance USD","id":"binance-usd","e":1,"s":1,"g":0,"esg":2},
    {"name":"SushiSwap","id":"sushi","e":1,"s":1,"g":1,"esg":3},
    {"name":"PancakeSwap","id":"pancakeswap-token","e":1,"s":1,"g":1,"esg":3},
    {"name":"THORChain","id":"thorchain","e":1,"s":1,"g":0,"esg":2},
    {"name":"BitDAO","id":"bitdao","e":1,"s":1,"g":1,"esg":3},
    {"name":"Lido DAO","id":"lido-dao","e":1,"s":1,"g":1,"esg":3},
    {"name":"Optimism","id":"optimism","e":1,"s":1,"g":1,"esg":3},
    {"name":"Arbitrum","id":"arbitrum","e":1,"s":1,"g":1,"esg":3},
    {"name":"Cronos","id":"cronos","e":1,"s":0,"g":0,"esg":1},
    {"name":"NEO","id":"neo","e":1,"s":1,"g":1,"esg":3},
    {"name":"BitTorrent","id":"bittorrent","e":1,"s":1,"g":0,"esg":2},
    {"name":"WEMIX","id":"wemix-token","e":1,"s":1,"g":1,"esg":3},
    {"name":"Verge","id":"verge","e":0,"s":1,"g":0,"esg":1},
    {"name":"Bitcoin SV","id":"bitcoin-sv","e":0,"s":0,"g":0,"esg":0},
    {"name":"Kaspa","id":"kaspa","e":0,"s":0,"g":0,"esg":0},
    {"name":"Ravencoin","id":"ravencoin","e":0,"s":1,"g":0,"esg":1},
    {"name":"DigiByte","id":"digibyte","e":0,"s":1,"g":0,"esg":1},
    {"name":"Siacoin","id":"siacoin","e":0,"s":1,"g":0,"esg":1},
    {"name":"QRL","id":"quantum-resistant-ledger","e":0,"s":1,"g":0,"esg":1},
    {"name":"Zano","id":"zano","e":1,"s":1,"g":0,"esg":2},
    {"name":"Vertcoin","id":"vertcoin","e":0,"s":1,"g":0,"esg":1},
    {"name":"Firo","id":"firo","e":0,"s":1,"g":1,"esg":2},
    {"name":"Namecoin","id":"namecoin","e":0,"s":1,"g":0,"esg":1},
    {"name":"Grin","id":"grin","e":0,"s":1,"g":0,"esg":1},
    {"name":"EOS","id":"eos","e":1,"s":1,"g":1,"esg":3},
    {"name":"Uniswap","id":"uniswap","e":1,"s":1,"g":1,"esg":3},
    {"name":"Dai","id":"dai","e":1,"s":1,"g":1,"esg":3},
    {"name":"Chainlink","id":"chainlink","e":1,"s":1,"g":0,"esg":2},
    {"name":"Aave","id":"aave","e":1,"s":1,"g":1,"esg":3},
    {"name":"Filecoin","id":"filecoin","e":1,"s":1,"g":0,"esg":2},
    {"name":"Sui","id":"sui","e":1,"s":1,"g":0,"esg":2},
    {"name":"Aptos","id":"aptos","e":1,"s":1,"g":1,"esg":3},
]


In [4]:
print(len(coins))

100


In [5]:
data = pd.DataFrame(columns=[
    "id", "name", "timestamp",
    "open", "high", "low", "close",
    "volume",
    "e", "s", "g", "esg"
])
import time
for coin in coins:
    coin_id = coin["id"]
    coin_name = coin["name"]

    success = False

    for attempt in range(1, 2):
        try:
            df = get_ohlc(coin_id, days="365")

            # ---- Inject metadata into every OHLC row ----
            df["id"] = coin_id
            df["name"] = coin_name
            df["e"] = coin["e"]
            df["s"] = coin["s"]
            df["g"] = coin["g"]
            df["esg"] = coin["esg"]

            # Ensure column order consistency
            df = df[
                ["id", "name", "timestamp",
                 "open", "high", "low", "close",
                 "volume", "e", "s", "g", "esg"]
            ]

            data = pd.concat([data, df], ignore_index=True)

            print(f"[OK] {coin_id} | rows={len(df)}")
            success = True
            break

        except Exception as e:
            print(f"[WARN] {coin_id} failed attempt {attempt}: {e}")
            time.sleep(10)  # backoff on failure

    if not success:
        print(f"[ERROR] Giving up on {coin_id}")

    # Rate limit protection between coins
    time.sleep(10)


C:\Users\caich\AppData\Local\Temp\ipykernel_31796\4203315635.py:33: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  data = pd.concat([data, df], ignore_index=True)


[OK] bitcoin | rows=365
[OK] ethereum | rows=365
[OK] cardano | rows=365
[OK] solana | rows=365
[OK] polkadot | rows=365
[OK] algorand | rows=365
[OK] tezos | rows=365
[OK] avalanche-2 | rows=365
[OK] near | rows=365
[OK] hedera-hashgraph | rows=365
[OK] stellar | rows=365
[OK] celo | rows=365
[OK] ripple | rows=365
[OK] nano | rows=365
[OK] iota | rows=365
[OK] energy-web-token | rows=365
[OK] power-ledger | rows=365
[OK] regen | rows=365
[OK] klima-dao | rows=365
[OK] moss-carbon-credit | rows=365
[OK] impt | rows=365
[OK] solarcoin | rows=365
[OK] peercoin | rows=365
[OK] flow | rows=365
[OK] wax | rows=365
[OK] vechain | rows=365
[OK] chia | rows=365
[OK] signum | rows=365
[OK] telcoin | rows=365
[OK] electroneum | rows=365
[OK] worldcoin-wld | rows=365
[OK] gooddollar | rows=365
[OK] vitadao | rows=365
[OK] civic | rows=365
[OK] decred | rows=365
[OK] ethereum-name-service | rows=365
[OK] cosmos | rows=365
[OK] kava | rows=365
[OK] gnosis | rows=365
[OK] maker | rows=365
[WARN] po

In [6]:
data

,id,name,timestamp,open,high,low,close,volume,e,s,g,esg
0,bitcoin,Bitcoin,2024-12-23,95094.273949,95094.273949,95094.273949,95094.273949,4.461902e+10,0,1,0,1
1,bitcoin,Bitcoin,2024-12-24,94644.910855,94644.910855,94644.910855,94644.910855,6.493779e+10,0,1,0,1
2,bitcoin,Bitcoin,2024-12-25,98695.714008,98695.714008,98695.714008,98695.714008,4.916909e+10,0,1,0,1
3,bitcoin,Bitcoin,2024-12-26,99344.954174,99344.954174,99344.954174,99344.954174,3.396375e+10,0,1,0,1
4,bitcoin,Bitcoin,2024-12-27,95678.312446,95678.312446,95678.312446,95678.312446,4.504934e+10,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...
34305,aptos,Aptos,2025-12-18,1.499948,1.499948,1.499948,1.499948,2.072894e+08,1,1,1,3
34306,aptos,Aptos,2025-12-19,1.450820,1.450820,1.450820,1.450820,1.359078e+08,1,1,1,3
34307,aptos,Aptos,2025-12-20,1.637996,1.637996,1.637996,1.637996,1.506050e+08,1,1,1,3
34308,aptos,Aptos,2025-12-21,1.611744,1.611744,1.611744,1.611744,2.527610e+08,1,1,1,3


In [9]:
data['name'].nunique()

100

In [8]:
new = [
    {"name":"Polygon","id":"polygon-bridged-wbtc-polygon-pos","e":1,"s":1,"g":1,"esg":3},
    {"name":"SocialGood","id":"social-good-project","e":1,"s":1,"g":0,"esg":2},
    {"name":"Terra Classic","id":"luna-wormhole","e":1,"s":0,"g":1,"esg":2},
    {"name":"Cronos","id":"crypto-com-chain","e":1,"s":0,"g":0,"esg":1},
    {"name":"Bitcoin SV","id":"bitcoin-cash-sv","e":0,"s":0,"g":0,"esg":0},
    {"name":"Firo","id":"zcoin","e":0,"s":1,"g":1,"esg":2},
]


for coin in new:
    coin_id = coin["id"]
    coin_name = coin["name"]

    success = False

    for attempt in range(1, 2):
        try:
            df = get_ohlc(coin_id, days="365")

            # ---- Inject metadata into every OHLC row ----
            df["id"] = coin_id
            df["name"] = coin_name
            df["e"] = coin["e"]
            df["s"] = coin["s"]
            df["g"] = coin["g"]
            df["esg"] = coin["esg"]

            # Ensure column order consistency
            df = df[
                ["id", "name", "timestamp",
                 "open", "high", "low", "close",
                 "volume", "e", "s", "g", "esg"]
            ]

            data = pd.concat([data, df], ignore_index=True)

            print(f"[OK] {coin_id} | rows={len(df)}")
            success = True
            break

        except Exception as e:
            print(f"[WARN] {coin_id} failed attempt {attempt}: {e}")
            time.sleep(10)  # backoff on failure

    if not success:
        print(f"[ERROR] Giving up on {coin_id}")

    # Rate limit protection between coins
    time.sleep(10)

[OK] polygon-bridged-wbtc-polygon-pos | rows=365
[OK] social-good-project | rows=365
[OK] luna-wormhole | rows=365
[OK] crypto-com-chain | rows=365
[OK] bitcoin-cash-sv | rows=365
[OK] zcoin | rows=365


In [10]:
data

,id,name,timestamp,open,high,low,close,volume,e,s,g,esg
0,bitcoin,Bitcoin,2024-12-23,95094.273949,95094.273949,95094.273949,95094.273949,4.461902e+10,0,1,0,1
1,bitcoin,Bitcoin,2024-12-24,94644.910855,94644.910855,94644.910855,94644.910855,6.493779e+10,0,1,0,1
2,bitcoin,Bitcoin,2024-12-25,98695.714008,98695.714008,98695.714008,98695.714008,4.916909e+10,0,1,0,1
3,bitcoin,Bitcoin,2024-12-26,99344.954174,99344.954174,99344.954174,99344.954174,3.396375e+10,0,1,0,1
4,bitcoin,Bitcoin,2024-12-27,95678.312446,95678.312446,95678.312446,95678.312446,4.504934e+10,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...
36495,zcoin,Firo,2025-12-18,1.574282,1.574282,1.574282,1.574282,5.261014e+05,0,1,1,2
36496,zcoin,Firo,2025-12-19,1.416434,1.416434,1.416434,1.416434,5.409415e+05,0,1,1,2
36497,zcoin,Firo,2025-12-20,1.395952,1.395952,1.395952,1.395952,5.114128e+05,0,1,1,2
36498,zcoin,Firo,2025-12-21,1.358657,1.358657,1.358657,1.358657,4.908971e+05,0,1,1,2


In [11]:
data.to_csv('cryptos')